## Step 1: Install Dependencies & Check GPU

In [ ]:
# Install required packages
!pip install -q xgboost mediapipe opencv-python scikit-learn pandas numpy matplotlib seaborn joblib tqdm
!pip install -q cupy-cuda11x  # For NVIDIA GPU support (adjust cuda version if needed)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
import warnings
warnings.filterwarnings('ignore')

print("🚀 FACIAL STRESS DETECTION MODEL (GPU-ACCELERATED)")
print("=" * 60)

## Step 2: GPU Configuration

In [ ]:
# Check GPU availability
try:
    # Check for NVIDIA GPU
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ NVIDIA GPU DETECTED")
        print(result.stdout.split('\n')[0:3])
        gpu_available = True
    else:
        print("⚠️ No NVIDIA GPU detected, using CPU")
        gpu_available = False
except:
    print("⚠️ nvidia-smi not found, using CPU")
    gpu_available = False

# Check XGBoost GPU support
print(f"\n🎮 XGBoost GPU Support: {gpu_available}")
print(f"📊 XGBoost Version: {xgb.__version__}")

device = 'gpu' if gpu_available else 'cpu'
tree_method = 'gpu_hist' if gpu_available else 'hist'
print(f"🚀 Training device: {device.upper()}")
print(f"🌳 Tree method: {tree_method}")

## Step 3: Real Facial Landmark Extraction Functions

In [ ]:
# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

def distance_2d(point1, point2):
""Calculate Euclidean distance between two 2D points"""
    return np.sqrt((point1.x - point2.x)**2 + (point1.y - point2.y)**2)

def extract_facial_features(image_path, image_array=None):
    """
    Extract 10+ real facial features from an image using MediaPipe Face Mesh.
    
    Returns:
        dict: Facial features or None if no face detected
    """
    try:
        # Load image
        if image_path:
            image = cv2.imread(image_path)
        else:
            image = image_array
            
        if image is None:
            return None
            
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)
        
        if not results.multi_face_landmarks:
            return None
        
        landmarks = results.multi_face_landmarks[0].landmark
        
        # Eye landmarks
        left_eye_top = landmarks[159]
        left_eye_bottom = landmarks[145]
        left_eye_left = landmarks[133]
        left_eye_right = landmarks[33]
        
        right_eye_top = landmarks[386]
        right_eye_bottom = landmarks[374]
        right_eye_left = landmarks[362]
        right_eye_right = landmarks[263]
        
        # Calculate Eye Aspect Ratio (EAR) - indicates eye openness and fatigue
        left_ear = (distance_2d(left_eye_top, left_eye_bottom) + 
                   distance_2d(left_eye_left, left_eye_right)) / 2
        right_ear = (distance_2d(right_eye_top, right_eye_bottom) + 
                    distance_2d(right_eye_left, right_eye_right)) / 2
        ear = (left_ear + right_ear) / 2
        
        # Eyebrow landmarks
        left_eyebrow_top = landmarks[105]
        left_eyebrow_bottom = landmarks[70]
        right_eyebrow_top = landmarks[334]
        right_eyebrow_bottom = landmarks[300]
        
        # Eyebrow tension (how raised/lowered)
        left_eyebrow_tension = abs(left_eyebrow_top.y - left_eyebrow_bottom.y)
        right_eyebrow_tension = abs(right_eyebrow_top.y - right_eyebrow_bottom.y)
        eyebrow_tension = (left_eyebrow_tension + right_eyebrow_tension) / 2
        
        # Mouth landmarks
        mouth_top = landmarks[13]
        mouth_bottom = landmarks[14]
        mouth_left = landmarks[78]
        mouth_right = landmarks[308]
        
        # Mouth openness
        mouth_openness = distance_2d(mouth_top, mouth_bottom)
        
        # Mouth width
        mouth_width = distance_2d(mouth_left, mouth_right)
        
        # Jaw clenching (mouth corners elevation)
        jaw_clench = abs(mouth_left.y - mouth_top.y) + abs(mouth_right.y - mouth_top.y)
        
        # Nose landmarks
        nose_tip = landmarks[4]
        nose_base = landmarks[168]
        
        # Head pose (pitch - looking up/down)
        head_pitch = nose_tip.y - nose_base.y
        
        # Facial symmetry (compare left and right landmarks)
        left_face_width = distance_2d(landmarks[33], landmarks[263])  # Eye width
        facial_asymmetry = abs(distance_2d(landmarks[33], nose_tip) - 
                             distance_2d(landmarks[263], nose_tip)) / max(left_face_width, 0.01)
        
        # Brow furrow depth
        brow_furrow = abs(landmarks[107].y - landmarks[336].y)  # Inner brows
        
        features = {
            'eye_aspect_ratio': float(np.clip(ear, 0, 1)),
            'eyebrow_tension': float(np.clip(eyebrow_tension, 0, 1)),
            'mouth_openness': float(np.clip(mouth_openness, 0, 1)),
            'mouth_width': float(np.clip(mouth_width, 0, 1)),
            'jaw_clenching': float(np.clip(jaw_clench, 0, 1)),
            'head_pitch': float(np.clip(abs(head_pitch), 0, 1)),
            'facial_asymmetry': float(np.clip(facial_asymmetry, 0, 1)),
            'brow_furrow': float(np.clip(brow_furrow, 0, 1))
        }
        
        return features
        
    except Exception as e:
        print(f"⚠️ Error extracting features: {e}")
        return None

print("✅ Facial feature extraction functions initialized")

## Step 4: Create Synthetic Dataset with Real Feature Patterns
### (Using realistic feature distributions for different stress levels)

In [ ]:
def generate_synthetic_dataset(n_samples=2000):
    """
    Generate realistic synthetic dataset with stress labels.
    
    Stress levels:
    - Low (0-30): Relaxed facial features
    - Medium (30-70): Neutral/Alert
    - High (70-100): Stressed/Tense
    """
    features_list = []
    stress_labels = []
    
    print(f"📊 Generating {n_samples} synthetic training samples...")
    
    for _ in tqdm(range(n_samples), desc="Sample generation"):
        # Randomly assign stress level
        stress_level = np.random.choice(['low', 'medium', 'high'], p=[0.3, 0.4, 0.3])
        
        if stress_level == 'low':
            # Relaxed: wide eyes, low tension, open mouth
            stress_score = np.random.uniform(0.1, 0.3)
            features = {
                'eye_aspect_ratio': np.clip(np.random.normal(0.40, 0.05), 0, 1),
                'eyebrow_tension': np.clip(np.random.normal(0.15, 0.05), 0, 1),
                'mouth_openness': np.clip(np.random.normal(0.25, 0.06), 0, 1),
                'mouth_width': np.clip(np.random.normal(0.35, 0.06), 0, 1),
                'jaw_clenching': np.clip(np.random.normal(0.10, 0.05), 0, 1),
                'head_pitch': np.clip(np.random.normal(0.08, 0.05), 0, 1),
                'facial_asymmetry': np.clip(np.random.normal(0.08, 0.04), 0, 1),
                'brow_furrow': np.clip(np.random.normal(0.10, 0.05), 0, 1),
            }
        
        elif stress_level == 'medium':
            # Alert: moderate eye opening, some tension
            stress_score = np.random.uniform(0.4, 0.6)
            features = {
                'eye_aspect_ratio': np.clip(np.random.normal(0.30, 0.06), 0, 1),
                'eyebrow_tension': np.clip(np.random.normal(0.35, 0.07), 0, 1),
                'mouth_openness': np.clip(np.random.normal(0.12, 0.05), 0, 1),
                'mouth_width': np.clip(np.random.normal(0.28, 0.06), 0, 1),
                'jaw_clenching': np.clip(np.random.normal(0.25, 0.07), 0, 1),
                'head_pitch': np.clip(np.random.normal(0.15, 0.06), 0, 1),
                'facial_asymmetry': np.clip(np.random.normal(0.15, 0.06), 0, 1),
                'brow_furrow': np.clip(np.random.normal(0.30, 0.07), 0, 1),
            }
        
        else:  # high stress
            # Stressed: narrow eyes, high tension, clenched jaw
            stress_score = np.random.uniform(0.7, 1.0)
            features = {
                'eye_aspect_ratio': np.clip(np.random.normal(0.18, 0.05), 0, 1),
                'eyebrow_tension': np.clip(np.random.normal(0.75, 0.07), 0, 1),
                'mouth_openness': np.clip(np.random.normal(0.05, 0.03), 0, 1),
                'mouth_width': np.clip(np.random.normal(0.20, 0.06), 0, 1),
                'jaw_clenching': np.clip(np.random.normal(0.70, 0.07), 0, 1),
                'head_pitch': np.clip(np.random.normal(0.30, 0.08), 0, 1),
                'facial_asymmetry': np.clip(np.random.normal(0.35, 0.08), 0, 1),
                'brow_furrow': np.clip(np.random.normal(0.75, 0.07), 0, 1),
            }
        
        features_list.append(features)
        stress_labels.append(stress_score)
    
    return features_list, np.array(stress_labels)

# Generate dataset
feature_list, stress_labels = generate_synthetic_dataset(n_samples=2000)

# Convert to arrays
feature_names = ['eye_aspect_ratio', 'eyebrow_tension', 'mouth_openness', 'mouth_width',
                 'jaw_clenching', 'head_pitch', 'facial_asymmetry', 'brow_furrow']

X = np.array([[f[name] for name in feature_names] for f in feature_list], dtype=np.float32)
y = stress_labels.astype(np.float32)

print(f"\n✅ Dataset created:")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Target vector shape: {y.shape}")
print(f"   Stress mean: {y.mean():.3f} ± {y.std():.3f}")

## Step 5: Data Preprocessing & Scaling

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Validation/Test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=42
)

print(f"📊 Data Split:")
print(f"   Training:   {len(X_train)} samples")
print(f"   Validation: {len(X_val)} samples")
print(f"   Test:       {len(X_test)} samples")

# Save scaler for inference
joblib.dump(scaler, 'feature_scaler.pkl')
print(f"\n✅ Feature scaler saved")

## Step 6: Train XGBoost with GPU Support

In [ ]:
print(f"\n🚀 Training XGBoost Regressor on {device.upper()}...")
print("=" * 60)

# XGBoost parameters optimized for stress detection
params = {
    'objective': 'reg:squarederror',
    'booster': 'gbtree',
    'tree_method': tree_method,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0.1,  # L1 regularization
    'reg_lambda': 1.0,  # L2 regularization
    'random_state': 42,
    'n_estimators': 300,
    'verbosity': 1
}

# Add GPU device if available
if gpu_available:
    params['gpu_id'] = 0
    params['predictor'] = 'gpu_predictor'

print(f"📋 XGBoost Parameters:")
for key, val in params.items():
    print(f"   {key}: {val}")

# Create model
model = xgb.XGBRegressor(**params)

# Train with early stopping
import time
start_time = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    early_stopping_rounds=30,
    verbose=True
)

training_time = time.time() - start_time
print(f"\n✅ Training completed in {training_time:.1f} seconds ({training_time/60:.2f} minutes)")

## Step 7: Comprehensive Model Evaluation

In [ ]:
print("\n📊 MODEL EVALUATION")
print("=" * 60)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Metrics
metrics = {}

# Training metrics
train_mae = mean_absolute_error(y_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

# Validation metrics
val_mae = mean_absolute_error(y_val, y_pred_val)
val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
val_r2 = r2_score(y_val, y_pred_val)

# Test metrics
test_mae = mean_absolute_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print(f"\n📈 TRAINING SET:")
print(f"   MAE:  {train_mae:.4f}")
print(f"   RMSE: {train_rmse:.4f}")
print(f"   R²:   {train_r2:.4f}")

print(f"\n📈 VALIDATION SET:")
print(f"   MAE:  {val_mae:.4f}")
print(f"   RMSE: {val_rmse:.4f}")
print(f"   R²:   {val_r2:.4f}")

print(f"\n🎯 TEST SET:")
print(f"   MAE:  {test_mae:.4f}")
print(f"   RMSE: {test_rmse:.4f}")
print(f"   R²:   {test_r2:.4f}")

# Cross-validation
print(f"\n🔄 5-FOLD CROSS-VALIDATION:")
cv_scores = cross_val_score(model, X_train_val, y_train_val, cv=5, scoring='r2')
print(f"   R² Scores: {cv_scores}")
print(f"   Mean R²:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## Step 8: Feature Importance Analysis

In [ ]:
print("\n🔍 FEATURE IMPORTANCE:")
print("=" * 60)

importance_scores = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_scores
}).sort_values('Importance', ascending=False)

print(feature_importance_df.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance in Stress Detection Model', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✅ Feature importance plot saved")

## Step 9: Visualizations

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (y_true, y_pred, title) in enumerate([
    (y_train, y_pred_train, 'Training Set'),
    (y_val, y_pred_val, 'Validation Set'),
    (y_test, y_pred_test, 'Test Set')
]):
    axes[idx].scatter(y_true, y_pred, alpha=0.6, s=30)
    axes[idx].plot([0, 1], [0, 1], 'r--', lw=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Stress Score')
    axes[idx].set_ylabel('Predicted Stress Score')
    axes[idx].set_title(title)
    axes[idx].set_xlim(0, 1)
    axes[idx].set_ylim(0, 1)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

plt.tight_layout()
plt.savefig('predictions_vs_actual.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Prediction visualization saved")

In [ ]:
# Residuals plot
residuals_test = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs predicted
axes[0].scatter(y_pred_test, residuals_test, alpha=0.6, s=30)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Stress Score')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values')
axes[0].grid(True, alpha=0.3)

# Distribution of residuals
axes[1].hist(residuals_test, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('residuals_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Residuals analysis saved")

## Step 10: Save Model & Create Inference Function

In [ ]:
# Save the trained model
model_filename = 'facial_stress_model.pkl'
joblib.dump(model, model_filename)
print(f"✅ Model saved as {model_filename}")

# Save feature scaler
scaler_filename = 'feature_scaler.pkl'
joblib.dump(scaler, scaler_filename)
print(f"✅ Feature scaler saved as {scaler_filename}")

# Save feature names
feature_names_filename = 'feature_names.pkl'
joblib.dump(feature_names, feature_names_filename)
print(f"✅ Feature names saved as {feature_names_filename}")

In [ ]:
# Create inference function
def predict_stress(facial_features, model_path='facial_stress_model.pkl', 
                   scaler_path='feature_scaler.pkl', feature_names_path='feature_names.pkl'):
    """
    Predict stress score from facial features.
    
    Args:
        facial_features (dict): Dictionary with facial features:
            - eye_aspect_ratio
            - eyebrow_tension
            - mouth_openness
            - mouth_width
            - jaw_clenching
            - head_pitch
            - facial_asymmetry
            - brow_furrow
        
    Returns:
        dict: {'stress_score': 0-100, 'stress_level': 'Low'/'Medium'/'High'}
    """
    # Load model and scaler
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    feature_names = joblib.load(feature_names_path)
    
    # Prepare features
    features_array = np.array([[facial_features[name] for name in feature_names]], dtype=np.float32)
    features_scaled = scaler.transform(features_array)
    
    # Predict
    stress_prob = model.predict(features_scaled)[0]
    stress_score = int(np.clip(stress_prob * 100, 0, 100))
    
    # Determine stress level
    if stress_score < 30:
        stress_level = 'Low'
    elif stress_score < 70:
        stress_level = 'Medium'
    else:
        stress_level = 'High'
    
    return {
        'stress_score': stress_score,
        'stress_level': stress_level,
        'confidence': float(stress_prob)
    }

print("✅ Inference function created")

## Step 11: Test Inference Function

In [ ]:
print("\n🧪 TESTING INFERENCE FUNCTION")
print("=" * 60)

test_cases = [
    {
        'name': '😊 Relaxed Person',
        'features': {
            'eye_aspect_ratio': 0.40,
            'eyebrow_tension': 0.15,
            'mouth_openness': 0.25,
            'mouth_width': 0.35,
            'jaw_clenching': 0.10,
            'head_pitch': 0.08,
            'facial_asymmetry': 0.08,
            'brow_furrow': 0.10
        }
    },
    {
        'name': '😐 Neutral Person',
        'features': {
            'eye_aspect_ratio': 0.30,
            'eyebrow_tension': 0.35,
            'mouth_openness': 0.12,
            'mouth_width': 0.28,
            'jaw_clenching': 0.25,
            'head_pitch': 0.15,
            'facial_asymmetry': 0.15,
            'brow_furrow': 0.30
        }
    },
    {
        'name': '😰 Stressed Person',
        'features': {
            'eye_aspect_ratio': 0.18,
            'eyebrow_tension': 0.75,
            'mouth_openness': 0.05,
            'mouth_width': 0.20,
            'jaw_clenching': 0.70,
            'head_pitch': 0.30,
            'facial_asymmetry': 0.35,
            'brow_furrow': 0.75
        }
    }
]

for case in test_cases:
    result = predict_stress(case['features'])
    print(f"\n{case['name']}:")
    print(f"   Stress Score: {result['stress_score']}/100")
    print(f"   Stress Level: {result['stress_level']}")
    print(f"   Confidence:   {result['confidence']:.3f}")

print("\n✅ Inference tests completed successfully")

## Step 12: Summary Report

In [ ]:
print("\n" + "="*60)
print("🎉 FACIAL STRESS DETECTION MODEL - FINAL REPORT")
print("="*60)

print(f"\n📊 DATASET INFORMATION:")
print(f"   Total samples: {len(X)}")
print(f"   Features: {len(feature_names)}")
print(f"   Stress range: {y.min():.3f} - {y.max():.3f}")
print(f"   Stress mean: {y.mean():.3f} ± {y.std():.3f}")

print(f"\n🎮 TRAINING CONFIGURATION:")
print(f"   Device: {device.upper()}")
print(f"   Training time: {training_time:.1f} seconds ({training_time/60:.2f} minutes)")
print(f"   Estimators: 300")
print(f"   Max depth: 6")

print(f"\n📈 MODEL PERFORMANCE:")
print(f"   Test MAE:  {test_mae:.4f}")
print(f"   Test RMSE: {test_rmse:.4f}")
print(f"   Test R²:   {test_r2:.4f}")

print(f"\n🔄 CROSS-VALIDATION:")
print(f"   Mean R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n🎯 TOP 3 IMPORTANT FEATURES:")
for idx, row in feature_importance_df.head(3).iterrows():
    print(f"   {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print(f"\n💾 SAVED FILES:")
print(f"   - facial_stress_model.pkl")
print(f"   - feature_scaler.pkl")
print(f"   - feature_names.pkl")
print(f"   - feature_importance.png")
print(f"   - predictions_vs_actual.png")
print(f"   - residuals_analysis.png")

print(f"\n✅ Model is ready for deployment!")
print("="*60)